In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

## Bước 2 — Lấy URL để thử

In [ ]:
# Ô 2 — Lấy URL độc để thử T1–T7 + danh sách domain sạch (để kiểm tra hợp lệ)
import glob, re, random
import pandas as pd
import tldextract
EXTRACT = tldextract.TLDExtract(suffix_list_urls=())

def reg_domain_of_host(host):
    ext = EXTRACT(host)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

A_path = f'{ROOT}/data/processed/A.parquet'
iscx = sorted(glob.glob(f'{ROOT}/data/raw/ISCX-URL2016/**/*.csv', recursive=True))
pick = lambda kw: [p for p in iscx if kw in os.path.basename(p).lower() and 'all' not in os.path.basename(p).lower()]

if os.path.exists(A_path):                                   # cách 1: Trí đã làm xong A.parquet
    A = pd.read_parquet(A_path)
    mal = A[(A.label == 1) & (A.split == 'train')]['url_norm']
    benign_domains = set(A[A.label == 0]['reg_domain'])
    SOURCE = 'A.parquet'
elif pick('phishing') and pick('benign'):                    # cách 2: đọc thẳng file gốc ISCX
    rd = lambda p: pd.read_csv(p, header=None, usecols=[0], encoding='latin-1')[0].astype(str)
    strip = lambda s: re.sub(r'^[a-zA-Z]+://', '', s.strip()).removeprefix('www.')
    mal = rd(pick('phishing')[0]).map(strip)
    benign_domains = set(rd(pick('benign')[0]).map(lambda u: reg_domain_of_host(strip(u).split('/')[0].lower())))
    SOURCE = 'ISCX gốc'
else:                                                        # cách 3: URL giả để code chạy được ngay
    mal = pd.Series(['account-blog16.top/login/verify.php?id=7', 'secure-paypal-update.example.net/webscr',
                     'mail.bank-confirm.example.org/signin', '203.0.113.9/wallet/update',
                     'apple-id-unlock.example.com/password/reset'] * 10)
    benign_domains = {'example.com', 'wikipedia.org', 'google.com'}
    SOURCE = 'URL giả (demo)'

N = 50
mal_urls = mal.sample(min(N, len(mal)), random_state=0).tolist()
print(f'Nguồn URL: {SOURCE} | {len(mal_urls)} URL độc | {len(benign_domains)} domain sạch')

In [ ]:
# Ô 3 — Ghi file src/attacks.py (TV3 sở hữu): 7 biến đổi T1–T7 + kiểm tra hợp lệ
ATTACKS_CODE = r"""
import re, random
from urllib.parse import urlsplit, urlunsplit
import tldextract
EXTRACT = tldextract.TLDExtract(suffix_list_urls=())

FREE_OPS, PAID_OPS = ['T1', 'T2', 'T3', 'T4'], ['T5', 'T6', 'T7']
PARAMS = [('utm_source', 'mail'), ('utm_medium', 'email'), ('ref', 'home'), ('lang', 'en'), ('page', '1')]
NEUTRAL_SUBS = ['account', 'support', 'app', 'portal', 'my', 'service', 'home', 'm']
NEUTRAL_WORDS = {'login': 'member', 'signin': 'portal', 'verify': 'profile', 'secure': 'home',
                 'account': 'user', 'update': 'news', 'confirm': 'continue', 'password': 'settings',
                 'webscr': 'page', 'billing': 'orders', 'wallet': 'items', 'bank': 'center'}
HOMOGLYPH = {'a': 'а', 'e': 'е', 'o': 'о', 'p': 'р', 'c': 'с', 'x': 'х', 'y': 'у', 'i': 'і'}  # chữ Cyrillic
TYPO = {'l': '1', 'o': '0', 'i': 'l', 'e': '3', 's': '5', 'a': '4', 'm': 'rn'}

def reg_domain_of_host(host):
    ext = EXTRACT(host)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

def to_attack_form(url_norm):
    return 'http://' + url_norm          # URL kẻ tấn công gửi đi luôn có scheme

def _host_parts(url):
    p = urlsplit(url)
    return p, EXTRACT(p.hostname or '')

def _rebuild(p, sub, dom, suf):
    host = '.'.join(x for x in [sub, dom, suf] if x)
    return urlunsplit((p.scheme, host, p.path, p.query, ''))

def t1_add_param(url, rng):          # T1: thêm tham số trông bình thường (miễn phí)
    p = urlsplit(url); k, v = rng.choice(PARAMS)
    q = (p.query + '&' if p.query else '') + f'{k}={v}'
    return urlunsplit((p.scheme, p.netloc, p.path or '/', q, ''))

def t2_add_subdomain(url, rng):      # T2: thêm subdomain (miễn phí, wildcard DNS)
    p, ext = _host_parts(url)
    if not ext.suffix: return None
    sub = rng.choice(NEUTRAL_SUBS) + ('.' + ext.subdomain if ext.subdomain else '')
    return _rebuild(p, sub, ext.domain, ext.suffix)

def t3_neutral_word(url, rng):       # T3: thay từ khóa đáng ngờ bằng từ trung tính (miễn phí)
    p = urlsplit(url); low = (p.path + '?' + p.query).lower()
    hits = [w for w in NEUTRAL_WORDS if w in low]
    if not hits: return None
    w = rng.choice(hits)
    path = re.sub(w, NEUTRAL_WORDS[w], p.path, flags=re.I)
    query = re.sub(w, NEUTRAL_WORDS[w], p.query, flags=re.I)
    return urlunsplit((p.scheme, p.netloc, path, query, ''))

def t4_https(url, rng):              # T4: http -> https (gần như miễn phí)
    return 'https://' + url[len('http://'):] if url.startswith('http://') else None

def t5_hyphen(url, rng):             # T5: chèn gạch nối vào domain (phải mua domain mới)
    p, ext = _host_parts(url); d = ext.domain
    if not ext.suffix or len(d) < 4 or d.startswith('xn--'): return None
    i = rng.randint(2, len(d) - 2)
    if '-' in (d[i - 1], d[i]): return None
    return _rebuild(p, ext.subdomain, d[:i] + '-' + d[i:], ext.suffix)

def t6_typo(url, rng):               # T6: typosquatting (phải mua domain mới)
    p, ext = _host_parts(url); d = ext.domain
    idx = [i for i, ch in enumerate(d) if ch in TYPO]
    if not ext.suffix or not idx or d.startswith('xn--'): return None
    i = rng.choice(idx)
    return _rebuild(p, ext.subdomain, d[:i] + TYPO[d[i]] + d[i + 1:], ext.suffix)

def t7_homoglyph(url, rng):          # T7: ký tự Cyrillic giống chữ Latin, lưu dạng punycode (phải mua domain)
    p, ext = _host_parts(url); d = ext.domain
    idx = [i for i, ch in enumerate(d) if ch in HOMOGLYPH]
    if not ext.suffix or not idx or d.startswith('xn--'): return None
    i = rng.choice(idx)
    uni = d[:i] + HOMOGLYPH[d[i]] + d[i + 1:]
    try:
        puny = uni.encode('idna').decode('ascii')
    except UnicodeError:
        return None
    return _rebuild(p, ext.subdomain, puny, ext.suffix)

TRANSFORMS = {'T1': t1_add_param, 'T2': t2_add_subdomain, 'T3': t3_neutral_word, 'T4': t4_https,
              'T5': t5_hyphen, 'T6': t6_typo, 'T7': t7_homoglyph}

def reg_domain_of_url(url):
    return reg_domain_of_host(urlsplit(url).hostname or '')

def is_valid(old, new, op, benign_domains):
    if new is None or len(new) > 2048: return False
    try:
        p = urlsplit(new)
    except ValueError:
        return False
    if not p.hostname: return False
    rd_old, rd_new = reg_domain_of_url(old), reg_domain_of_url(new)
    if op in FREE_OPS and rd_old != rd_new: return False          # T1–T4 giữ nguyên registered domain
    if op in PAID_OPS and rd_new in benign_domains: return False  # không "mượn" domain thật
    return True

def apply_op(op, url, rng):
    try:
        return TRANSFORMS[op](url, rng)
    except (ValueError, UnicodeError):
        return None

def defang(url):
    # Viết URL độc ở dạng vô hiệu hóa khi đưa vào báo cáo: hxxp, [.]
    return url.replace('http', 'hxxp', 1).replace('.', '[.]')
"""
with open(f'{ROOT}/src/attacks.py', 'w', encoding='utf-8') as f:
    f.write(ATTACKS_CODE)

import sys, importlib
src_dir = f'{ROOT}/src'
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()

import attacks
importlib.reload(attacks)
from attacks import *

rng = random.Random(0)
demo = to_attack_form('account-blog16.top/login/verify.php?id=7')
for op in TRANSFORMS:
    print(op, '->', apply_op(op, demo, random.Random(1)))

In [ ]:
# Ô 4 — Thử từng biến đổi trên 50 URL độc: bao nhiêu URL áp dụng được, bao nhiêu hợp lệ
rows = []
for op in TRANSFORMS:
    rng = random.Random(0)
    applied = valid = 0
    for u in mal_urls:
        old = to_attack_form(u)
        new = apply_op(op, old, rng)
        applied += new is not None
        valid += is_valid(old, new, op, benign_domains)
    rows.append({'T': op, 'loại': 'miễn phí' if op in FREE_OPS else 'tốn phí',
                 'áp dụng được': applied, 'hợp lệ': valid, 'tổng': len(mal_urls)})
summary = pd.DataFrame(rows)
summary.to_csv(f'{ROOT}/results/transform_check_w4.csv', index=False)
summary

In [ ]:
# Ô 5 — Xuất 20 mẫu hợp lệ cho mỗi biến đổi → results/samples_T1..T7.txt (dùng cho phụ lục báo cáo)
pool = mal.sample(min(2000, len(mal)), random_state=1).tolist()   # lấy rộng hơn để đủ 20 mẫu
for op in TRANSFORMS:
    rng, out = random.Random(0), []
    for u in pool:
        old = to_attack_form(u); new = apply_op(op, old, rng)
        if is_valid(old, new, op, benign_domains):
            out.append(f'{defang(old)}\n  -> {defang(new)}')
        if len(out) == 20: break
    with open(f'{ROOT}/results/samples_{op}.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(out))
    print(f'{op}: {len(out)} mẫu | ví dụ: {out[0] if out else "(không có)"}')

In [ ]:
# Ô 7 — (Nên có) Lưu snapshot feed OpenPhish hôm nay → nguồn C sau này. Chạy mỗi ngày 1 lần.
# CHỈ lưu chuỗi URL vào file text. Không mở, không bấm link nào.
import datetime, urllib.request
os.makedirs(f'{ROOT}/data/raw/openphish', exist_ok=True)
today = datetime.date.today().isoformat()
try:
    txt = urllib.request.urlopen('https://openphish.com/feed.txt', timeout=30).read().decode('utf-8', 'ignore')
    with open(f'{ROOT}/data/raw/openphish/{today}.txt', 'w') as f:
        f.write(txt)
    print(today, '-', len(txt.split()), 'URL đã lưu')
except Exception as e:
    print('Không tải được feed hôm nay:', e)

In [ ]:
kw = list(NEUTRAL_WORDS)
mal_all = A[A.label == 1]
in_path = mal_all['rest'].str.lower().apply(lambda r: any(w in r for w in kw))
in_host = mal_all['host'].str.lower().apply(lambda h: any(w in h for w in kw))
print(f'URL độc có từ khóa ở path/tham số: {in_path.mean()*100:.1f}%')
print(f'URL độc có từ khóa ở tên domain:   {in_host.mean()*100:.1f}%')
print(mal_all['url_norm'].sample(10, random_state=0).to_string())